# Kaggle Test Evaluation

Notebook nay dung de evaluate cac checkpoint da train tren tap `test-00000-of-00001.parquet` ngay tren Kaggle.

Notebook tu tai output tu 2 kernels `anhnguyen0812/nlp-finetune` va `anhnguyen0812/nlp-sumarization-causal-lm`, xu ly layout `summarization_outputs`, gom cac run co `resolved_config.json` va `best/*.safetensors`, roi chay test.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
RUNS_ROOT = WORKING / 'test_eval_runs'
KERNEL_OUTPUT_DIR = WORKING / 'kernel_outputs'
KAGGLE_KERNEL_OUTPUTS = [
    'anhnguyen0812/nlp-finetune',
    'anhnguyen0812/nlp-sumarization-causal-lm',
]
DOWNLOAD_KERNEL_OUTPUTS = True
OUT_DIR = WORKING / 'test_eval_outputs'
TEST_FILE = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet')
TEST_BASENAME = TEST_FILE.name
MAX_TEST_SAMPLES = None  # dat 50 de smoke test nhanh

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

WORKING.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
10422d5 Make Kaggle test output download more robust


0

In [2]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)
run('nvidia-smi', check=False)


CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.4 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

0

## Prepare Runs And Test File

Notebook tai 2 kernel output, xu ly dung 2 layout hien co: `summarization_outputs` cua notebook pretrained/causal va `report_experiment_outputs` cua notebook all-in-one. Moi run co `resolved_config.json` va `best/*.safetensors` se duoc gom vao `/kaggle/working/test_eval_runs`.


In [3]:
EXPECTED_OUTPUT_ROOT_NAMES = ['summarization_outputs', 'report_experiment_outputs']
RESULT_ZIP_NAMES = ['allinone_report_results.zip', 'summarization_results.zip', 'causal_lm_results.zip']

def find_files(root, pattern):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(pattern))

def download_kernel_outputs():
    if not DOWNLOAD_KERNEL_OUTPUTS:
        return
    KERNEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for kernel in KAGGLE_KERNEL_OUTPUTS:
        dest = KERNEL_OUTPUT_DIR / kernel.split('/')[-1]
        dest.mkdir(parents=True, exist_ok=True)
        if any(dest.iterdir()):
            print('KERNEL OUTPUT EXISTS:', kernel, dest)
            continue
        code = run(f'kaggle kernels output {kernel} -p {dest}', cwd=WORKING, check=False)
        if code != 0:
            print('WARN: could not download kernel output:', kernel)

def unzip_result_zips():
    zip_candidates = []
    search_roots = [KERNEL_OUTPUT_DIR, Path('/kaggle/input'), Path('/kaggle/working')]
    for root in search_roots:
        if not root.exists():
            continue
        for name in RESULT_ZIP_NAMES:
            zip_candidates.extend(root.rglob(name))
        zip_candidates.extend(p for p in root.rglob('*.zip') if p.name.startswith(('allinone_', 'summarization_', 'causal_lm_')))
    zip_candidates = sorted(set(p for p in zip_candidates if p.name != 'test_eval_results.zip'))
    if not zip_candidates:
        print('No result zip found. Will try direct kernel output folders.')
        return
    for zip_path in zip_candidates:
        print('UNZIP', zip_path, '->', WORKING)
        with ZipFile(zip_path) as zf:
            zf.extractall(WORKING)

def candidate_output_roots():
    roots = []
    search_roots = [KERNEL_OUTPUT_DIR, WORKING]
    if Path('/kaggle/input').exists():
        search_roots.append(Path('/kaggle/input'))
    for base in search_roots:
        if not base.exists():
            continue
        for root_name in EXPECTED_OUTPUT_ROOT_NAMES:
            direct = base / root_name
            if direct.exists():
                roots.append(direct)
            roots.extend(p for p in base.rglob(root_name) if p.is_dir())
    unique = []
    seen = set()
    for root in sorted(roots):
        key = str(root.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(root)
    return unique

def has_model_artifacts(run_dir):
    best = run_dir / 'best'
    if not best.exists():
        return False
    return any((best / name).exists() for name in ['adapter_model.safetensors', 'model.safetensors'])

def collect_run_dirs(output_roots):
    found = []
    for root in output_roots:
        for run_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if (run_dir / 'resolved_config.json').exists() and has_model_artifacts(run_dir):
                found.append(run_dir)
    unique = []
    seen = set()
    for run_dir in sorted(found):
        key = str(run_dir.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(run_dir)
    return unique

def build_merged_runs_root(run_dirs):
    if RUNS_ROOT.exists():
        shutil.rmtree(RUNS_ROOT)
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, run_dir in enumerate(run_dirs):
        name = run_dir.name
        dest = RUNS_ROOT / name
        if dest.exists():
            dest = RUNS_ROOT / f'{name}_{idx}'
        try:
            os.symlink(run_dir, dest, target_is_directory=True)
        except OSError:
            shutil.copytree(run_dir, dest)
        print('ADD RUN:', dest.name, '<-', run_dir)

def find_test_file():
    candidates = find_files('/kaggle/input', TEST_BASENAME) + find_files('/kaggle/working', TEST_BASENAME)
    if not candidates:
        raise FileNotFoundError(f'Khong thay {TEST_BASENAME}. Hay attach/upload Kaggle dataset co test parquet.')
    return candidates[0]

download_kernel_outputs()
unzip_result_zips()
output_roots = candidate_output_roots()
print('OUTPUT_ROOT_CANDIDATES:', output_roots)
run_dirs = collect_run_dirs(output_roots)
if not run_dirs:
    raise FileNotFoundError('Khong thay run folder nao co resolved_config.json va best/*.safetensors trong summarization_outputs/report_experiment_outputs.')
build_merged_runs_root(run_dirs)
TEST_FILE = find_test_file()
merged_run_dirs = sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir() and (p / 'resolved_config.json').exists())
print('TEST_FILE:', TEST_FILE)
print('RUNS_ROOT:', RUNS_ROOT)
print('RUNS:', [p.name for p in merged_run_dirs])


CMD: kaggle kernels output anhnguyen0812/nlp-finetune -p /kaggle/working/kernel_outputs/nlp-finetune
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/SAVE_VERSION_README.txt
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.git/HEAD
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.git/config
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.git/description
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.git/hooks/applypatch-msg.sample
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.git/hooks/commit-msg.sample
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.git/hooks/fsmonitor-watchman.sample
Output file downloaded to /kaggle/working/kernel_outputs/nlp-finetune/pretrained-summarization/.g

## Evaluate On Test

Mac dinh chay full test. Neu muon smoke test, sua `MAX_TEST_SAMPLES = 50` o cell dau.


In [4]:
args = f'--runs_root {RUNS_ROOT} --test_file {TEST_FILE} --out_dir {OUT_DIR}'
if MAX_TEST_SAMPLES:
    args += f' --max_test_samples {MAX_TEST_SAMPLES}'
run(f'{sys.executable} -u -m vn_summarization.evaluate_runs_on_test {args}', cwd=repo)


CMD: /usr/bin/python3 -u -m vn_summarization.evaluate_runs_on_test --runs_root /kaggle/working/test_eval_runs --test_file /kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet --out_dir /kaggle/working/test_eval_outputs
2026-06-10 16:28:57.552169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781108937.743115     101 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781108937.796547     101 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781108938.249579     101 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W00

0

In [5]:
zip_path = WORKING / 'test_eval_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUT_DIR.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
print('TEST CSV', OUT_DIR / 'test_results.csv')
print('TEST MD', OUT_DIR / 'test_results.md')
for file in sorted(files):
    print(file)


ZIP /kaggle/working/test_eval_results.zip
TEST CSV /kaggle/working/test_eval_outputs/test_results.csv
TEST MD /kaggle/working/test_eval_outputs/test_results.md
/kaggle/working/test_eval_outputs/bartpho_syllable_ep2_t4x2/predictions_test.jsonl
/kaggle/working/test_eval_outputs/bartpho_syllable_ep2_t4x2/resolved_test_config.json
/kaggle/working/test_eval_outputs/bartpho_syllable_ep2_t4x2/test_metrics.json
/kaggle/working/test_eval_outputs/bartpho_syllable_ep2_t4x2/validation_metrics.json
/kaggle/working/test_eval_outputs/best_test_run.json
/kaggle/working/test_eval_outputs/test_results.csv
/kaggle/working/test_eval_outputs/test_results.md
/kaggle/working/test_eval_outputs/vit5_news_warmstart_ep2_t4x2/predictions_test.jsonl
/kaggle/working/test_eval_outputs/vit5_news_warmstart_ep2_t4x2/resolved_test_config.json
/kaggle/working/test_eval_outputs/vit5_news_warmstart_ep2_t4x2/test_metrics.json
/kaggle/working/test_eval_outputs/vit5_news_warmstart_ep2_t4x2/validation_metrics.json
